# Episode 2 — JIT, Tracing, and the Jaxpr

**Student workbook** · code along with the video.

`jax.jit` traces your function once into a **jaxpr**, lowers to XLA, and caches the executable.

| | |
|---|---|
| **Chapter** | 1.2 · Part I — Pure JAX |
| **Prereq** | Episode 1 |
| **Next** | Episode 3 — automatic differentiation |

**JAX docs:** [Just-in-time compilation](https://docs.jax.dev/en/latest/jit-compilation.html) · [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [jaxpr](https://docs.jax.dev/en/latest/jaxpr.html) · [Control flow with JIT](https://docs.jax.dev/en/latest/control-flow.html) · [`jax.make_jaxpr`](https://docs.jax.dev/en/latest/_autosummary/jax.make_jaxpr.html) · [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html)


In [ ]:
# your code here


## How JAX transformations work

JAX reduces your function to a sequence of **primitives** — one fundamental op each. `make_jaxpr` shows that IR. Transformations only understand **side-effect-free** (pure) code: Python side effects are invisible to the jaxpr.

During tracing, arguments become **tracers** that record ops. Side effects like `append` still run during the trace, but they never appear in the compiled program. For debug output inside `jit`, use [`jax.debug.print`](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html) instead of `print`.

In [ ]:
# your code here


## `print` is a side effect — trace time only

`print` is not pure: it runs during tracing, but the jaxpr only captures array ops. The printed value is a tracer, not a concrete number.

In [ ]:
# your code here


## Python control flow is traced once per branch

A jaxpr captures the function **as executed on the arguments you pass**. A Python `if` on a **static** attribute like `ndim` picks one branch; the other branch is absent from the jaxpr.

In [ ]:
# your code here


## What `jit` does

Without `jit`, JAX dispatches one op at a time — XLA cannot fuse or optimize across your function. `jit` traces into a jaxpr, lowers to **StableHLO**, compiles once per cache key (shapes/dtypes/static args), and reuses the executable. New shapes → **retrace**.

Warm up before timing (first call pays compile cost). Use `block_until_ready()` because JAX dispatch is [asynchronous](https://docs.jax.dev/en/latest/async_dispatch.html).

## SELU: eager ops vs one fused kernel

In [ ]:
# your code here


## A 3-layer MLP to inspect

In [ ]:
# your code here


## `jit` wraps the jaxpr in a compiled call

In [ ]:
# your code here


## First call vs steady state

In [ ]:
# your code here


## Retrace on shape change

In [ ]:
# your code here


## Why you can't `jit` everything

Inside `jit`, traced array **values** cannot drive Python `if` / `while` — only **static** attributes like `shape` and `dtype` can. Condition on a runtime value → `TracerBoolConversionError`.

Fixes: rewrite with array ops (`jnp.where`), use [control-flow primitives](https://docs.jax.dev/en/latest/jax.lax.html#lax-control-flow) like `jax.lax.cond`, mark value-dependent args as **static**, or `jit` only the expensive inner body.

In [ ]:
# your code here


## Partial `jit` — compile the hot inner function

When Python control flow must stay outside `jit`, compile just the loop body. Define the jitted function **once** at module scope so the cache can hit.

In [ ]:
# your code here


## Static vs traced — what `jit` can and cannot specialize on

| Kind | Examples | At `jit` time |
|------|----------|----------------|
| **Traced** | `jnp` arrays | Shapes and dtypes are baked into the compiled program. New shapes → **retrace**. |
| **Static** | Python `int`, `bool`, `str`, `None` that control structure | Mark with `static_argnums` / `static_argnames`. Each distinct static value → **recompile**. |
| **Not compiled** | `print`, file I/O, mutation, arbitrary Python loops over Python lists | Run at **trace** time only (or not at all inside compiled code). |

**Rule of thumb:** tensor data → traced; hyperparameters and flags that pick branches → static (only when the set of values is small).

## `static_argnums` for Python scalars

In [ ]:
# your code here


## `static_argnames` and the decorator factory

Same idea as `static_argnums`, but by parameter name. With the decorator, pass static options to the factory: `@jax.jit(static_argnames=["n"])`.

In [ ]:
# your code here


## Print inside `jit` fires at **trace** time, not every run

In [ ]:
# your code here


## JIT caching — don't redefine functions in loops

`jax.jit` caches by function identity. Calling `jit` on a **new** function object each iteration (`partial`, `lambda`) forces recompilation every time. Reusing the same function object is fine.

In [ ]:
# your code here


> **Key insight:** JIT does not execute your Python code at runtime — it traces it once. Print statements inside jitted functions fire at trace time, not run time. This confusion catches everyone.

---
## Exercise

1. `make_jaxpr` on the 3-layer MLP — find a `dot_general` or `dot` primitive.
2. Show a side effect (`append`) that runs at trace time but is absent from the jaxpr.
3. Trigger `TracerBoolConversionError` with `if x > 0` inside `jit`.
4. Fix a value-dependent loop with `static_argnums` or `static_argnames`.
5. Change batch size and log how many distinct shapes your jitted function sees.
6. Time first vs steady-state `jit` call on the same shapes (warm up first).
7. Compare `jit(partial(f))` in a loop vs `jit(f)` — which recompiles?



In [ ]:
# your code here


**Next:** Episode 3 — `grad`, `value_and_grad`, and checkpointing.